# Twacha — Acne Severity Model Training

**Phase 3: Train MobileNetV2 on ACNE04 for acne severity classification**

This notebook:
1. Downloads ACNE04 from Hugging Face
2. Inspects label distribution (known to be imbalanced)
3. Fine-tunes MobileNetV2 with class-weighted loss
4. Validates on held-out test split
5. Exports model for backend integration

**Runtime:** Use GPU -> Runtime -> Change runtime type -> T4 GPU

## 1. Setup and Dependencies

In [ ]:
!pip install -q datasets torch torchvision timm pillow scikit-learn matplotlib seaborn onnx onnxruntime

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Download ACNE04 Dataset

**Sources available:**
- Hugging Face: ManuelHettich/acne04 (easiest for Colab)
- GitHub (original): https://github.com/xpwu95/LDL
- Roboflow Universe: search for public acne04 projects

ACNE04 has 4 severity grades (0-3):
- **0**: Mild (few lesions)
- **1**: Moderate
- **2**: Severe
- **3**: Very Severe

In [ ]:
from datasets import load_dataset

# Download from Hugging Face
print("Downloading ACNE04 from Hugging Face...")
ds = load_dataset("ManuelHettich/acne04")
print(f"Dataset loaded: {ds}")
print(f"Train samples: {len(ds['train'])}")
if 'test' in ds:
    print(f"Test samples: {len(ds['test'])}")

In [ ]:
# ========================================
# ALTERNATIVE: If Hugging Face does not work,
# download from GitHub (original source)
# ========================================
# !git clone https://github.com/xpwu95/LDL.git
# # Images will be in LDL/data/ACNE04/
# # Organize into severity grade folders manually

## 3. Inspect Label Distribution

Blueprint warns: ACNE04 has imbalanced severity distribution (mild cases dominate). Use class-weighting or augmentation during training, not raw accuracy as metric.

In [ ]:
# Inspect the dataset structure
sample = ds['train'][0]
print(f"Sample keys: {sample.keys()}")
print(f"Sample: {sample}")

# Determine the label column name
label_col = None
for col in ['label', 'severity', 'grade', 'class']:
    if col in sample:
        label_col = col
        break

if label_col is None:
    print("WARNING: Could not auto-detect label column.")
    print(f"Available columns: {list(sample.keys())}")
    label_col = input("Enter the label column name: ")

print(f"Using label column: '{label_col}'")

In [ ]:
# Count label distribution
labels = [item[label_col] for item in ds['train']]
label_counts = Counter(labels)
num_classes = len(label_counts)

print(f"Number of classes: {num_classes}")
print(f"Label distribution:")
for label, count in sorted(label_counts.items()):
    print(f"  Grade {label}: {count} images ({count/len(labels)*100:.1f}%)")

# Visualize distribution
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
grades = sorted(label_counts.keys())
counts = [label_counts[g] for g in grades]
colors = ['#4CAF50', '#FFC107', '#FF9800', '#F44336']
ax.bar([f'Grade {g}' for g in grades], counts, color=colors[:len(grades)])
ax.set_title('ACNE04 Severity Distribution (expect imbalance)')
ax.set_ylabel('Count')
for i, (g, c) in enumerate(zip(grades, counts)):
    ax.text(i, c + 5, str(c), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize sample images from each grade
fig, axes = plt.subplots(1, min(num_classes, 4), figsize=(16, 4))
if num_classes == 1:
    axes = [axes]

for i, grade in enumerate(sorted(label_counts.keys())[:4]):
    for item in ds['train']:
        if item[label_col] == grade:
            img = item['image'] if isinstance(item['image'], Image.Image) else Image.open(item['image'])
            axes[i].imshow(img)
            axes[i].set_title(f'Grade {grade}', fontsize=14)
            axes[i].axis('off')
            break

plt.suptitle('Sample Images per Severity Grade', fontsize=16)
plt.tight_layout()
plt.show()

## 4. Dataset and DataLoader Setup

In [ ]:
# Image transforms - MobileNetV2 expects 224x224, ImageNet normalization
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class AcneDataset(Dataset):
    def __init__(self, hf_dataset, label_col, transform=None):
        self.data = hf_dataset
        self.label_col = label_col
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img = item['image']
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        img = img.convert('RGB')
        label = item[self.label_col]
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
# Split into train/val/test (70/15/15)
from sklearn.model_selection import train_test_split

all_data = ds['train']
all_labels = [item[label_col] for item in all_data]
indices = list(range(len(all_data)))

train_idx, temp_idx = train_test_split(
    indices, test_size=0.3, stratify=all_labels, random_state=42
)
temp_labels = [all_labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, stratify=temp_labels, random_state=42
)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

train_data = all_data.select(train_idx)
val_data = all_data.select(val_idx)
test_data = all_data.select(test_idx)

train_dataset = AcneDataset(train_data, label_col, train_transform)
val_dataset = AcneDataset(val_data, label_col, val_transform)
test_dataset = AcneDataset(test_data, label_col, val_transform)

In [ ]:
# Class-weighted sampler to handle imbalanced classes
train_labels = [all_labels[i] for i in train_idx]
class_counts = Counter(train_labels)
total = len(train_labels)

class_weights = {cls: total / count for cls, count in class_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("Class weights (inverse frequency):")
for cls in sorted(class_weights):
    print(f"  Grade {cls}: {class_weights[cls]:.2f}")

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 5. Model - Fine-tune MobileNetV2

MobileNetV2 chosen for:
- Small model size (~3.4M params) - runs on mobile/edge
- Pre-trained on ImageNet - good feature extraction baseline
- Easy to export to ONNX for server inference

In [ ]:
def create_model(num_classes):
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

    # Freeze early layers, fine-tune later layers
    for param in model.features[:14].parameters():
        param.requires_grad = False

    # Replace classifier
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(model.last_channel, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, num_classes),
    )
    return model.to(device)

model = create_model(num_classes)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")

In [ ]:
# Loss with class weights + optimizer
weight_tensor = torch.tensor(
    [class_weights[i] for i in range(num_classes)],
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(weight=weight_tensor)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

## 6. Training Loop

In [ ]:
NUM_EPOCHS = 25
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # Validate
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_acne_model.pth')
        print(f'  New best model saved (val_acc: {val_acc:.4f})')

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train', linewidth=2)
ax1.plot(history['val_loss'], label='Val', linewidth=2)
ax1.set_title('Loss', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train', linewidth=2)
ax2.plot(history['val_acc'], label='Val', linewidth=2)
ax2.set_title('Accuracy', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Twacha Acne Model - Training Progress', fontsize=16)
plt.tight_layout()
plt.show()

## 7. Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_acne_model.pth'))
model.eval()

all_preds, all_true = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_true.extend(labels.numpy())

grade_names = [f'Grade {i}' for i in range(num_classes)]
print('TEST SET EVALUATION')
print('=' * 60)
print(classification_report(all_true, all_preds, target_names=grade_names))

cm = confusion_matrix(all_true, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=grade_names, yticklabels=grade_names, ax=ax)
ax.set_title('Confusion Matrix - Test Set', fontsize=14)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 8. Convert Grade to 0-10 Score

Twacha uses a continuous 0-10 scale (lower = healthier). We use softmax probabilities for a smooth score.

In [ ]:
def grade_to_score(logits, num_classes=4):
    probs = torch.softmax(logits, dim=-1)
    anchors = torch.tensor([1.5, 4.0, 7.0, 9.5], device=logits.device)[:num_classes]
    score = (probs * anchors).sum(dim=-1)
    return score.clamp(0.0, 10.0)

test_batch = next(iter(test_loader))
images, labels = test_batch[0][:5].to(device), test_batch[1][:5]

with torch.no_grad():
    logits = model(images)
    scores = grade_to_score(logits)

for i in range(5):
    print(f'  True grade: {labels[i].item()} -> Acne score: {scores[i].item():.1f}/10')

## 9. Export Model

In [ ]:
# Save PyTorch model
torch.save({
    'model_state_dict': model.state_dict(),
    'num_classes': num_classes,
    'input_size': 224,
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225],
    'grade_anchors': [1.5, 4.0, 7.0, 9.5],
}, 'twacha_acne_model.pth')
print('Saved: twacha_acne_model.pth')

# Export to ONNX
model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)
torch.onnx.export(
    model, dummy_input, 'twacha_acne_model.onnx',
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=13,
)
print('Saved: twacha_acne_model.onnx')

import onnx
onnx_model = onnx.load('twacha_acne_model.onnx')
onnx.checker.check_model(onnx_model)
print('ONNX model verified')

pth_size = os.path.getsize('twacha_acne_model.pth') / (1024*1024)
onnx_size = os.path.getsize('twacha_acne_model.onnx') / (1024*1024)
print(f'Model sizes: PyTorch={pth_size:.1f}MB, ONNX={onnx_size:.1f}MB')

In [ ]:
# Download the model files
from google.colab import files

print("Downloading trained model files...")
print("Place these in your backend/ directory, then update mock_cv.py")
files.download('twacha_acne_model.pth')
files.download('twacha_acne_model.onnx')

## 10. Integration with Twacha Backend

After training, place `twacha_acne_model.onnx` in the `backend/` directory.

Then in `app/mock_cv.py`, replace `_score_acne_heuristic()` with ONNX inference:

```python
import onnxruntime as ort

_ACNE_SESSION = None
def _get_acne_session():
    global _ACNE_SESSION
    if _ACNE_SESSION is None:
        model_path = os.path.join(os.path.dirname(__file__), '..', 'twacha_acne_model.onnx')
        if os.path.exists(model_path):
            _ACNE_SESSION = ort.InferenceSession(model_path)
    return _ACNE_SESSION

def _score_acne_model(img):
    session = _get_acne_session()
    if session is None:
        return _score_acne_heuristic(img)  # fallback
    
    resized = cv2.resize(img, (224, 224))
    blob = resized.astype(np.float32) / 255.0
    blob = (blob - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    blob = blob.transpose(2, 0, 1)[np.newaxis]
    
    logits = session.run(None, {'image': blob})[0]
    probs = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)
    anchors = np.array([1.5, 4.0, 7.0, 9.5])
    score = (probs * anchors).sum()
    return round(float(np.clip(score, 0, 10)), 1)
```

Add `onnxruntime` to `requirements.txt`.